In [86]:
print('hello !')

hello !


deciding architecture ---

user query --> classifier llm ---> classifies into one of [order_agent, refund_agent, payment_agent,
human_agent(this will be handoff)]

questions like --
i want a refund on thisn ---> for this make a method as a tool taking some rules and replies yes or no
I'm not able to pay ----> maybe an error code associated with the order placed
maybe a discount agent as well? ---> not sure but provides a random discount code to give 5-10% off on products
if query not related to any of this, then hand it over to human agent


classifier is easy

lets says user asks- where is my order - order123

instead of RAG, we SHOULD use DB lookup here. 

so, gpt suggests creating mock data for our usecase here. that is create mock json/python data and use get_order method as a tool in our order agent probably. more clarity to come as we start building this
---but this question is too vague, a real wont ask this question maybe ----need to think about this

maybe since this is V1, we are building the most basic things

user query --> classifier llm ---> classifies into one of [order_agent, refund_agent, payment_agent,
human_agent(this will be handoff)]

lets think abotu the jsom/python/dbn mock of orders :-
orders = {
    order123 = {
        orderDate : ...,
        total : ...,
        payment = {
            paymentStatus : failed/success/something else,
            statusCode : 
        },
        orderStatus : shipped/delivered/failed,
    }
}



GRAPH NODES

  query
    |
classifier
    |
1. order  2. refund                     3.payment    4.Human
    |          |                            |           |
  DBlookup    refundEligibleCheck()       dblookup      handoffToHumanAgwnrt(not sure on thjis)

Let's START !

In [87]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict) :
    messages : Annotated[list, add_messages]
    orderDetails : str
    refundDetails : str
    paymentInfo : str
    humanRequired : bool
    category : str

In [88]:
import json

with open('orders.json', 'r', encoding='utf-8') as order:
    orders = json.load(order)


In [89]:
from typing import Literal
from pydantic import BaseModel

#structured output for classifier llm
class classifier(BaseModel) : 
    category : Literal['order', 'refund', 'payment', 'human']

In [90]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

classifierLLM = ChatOpenAI(model='gpt-5-nano')
classifierLLM_with_SO = classifierLLM.with_structured_output(classifier)

In [91]:
#classifier agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def classifierAgent(old_state : State) :
    PROMPT = f''' 
    You are a classifier agent for an ecommerce platform. Your work is to analyze the incoming user queries and decide
    the category which they belong to. Your response should be the category the query belongs to.
    Here's the user query - {old_state['messages']}
    '''
    response = classifierLLM_with_SO.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(f'category : {response.category}')
    return {
        'category' : response.category
    }
    

In [92]:
def classify_route(old_state : State) :
    return 'order' if old_state['category']=='order' else 'refund' if old_state['category']=='refund' else 'payment' if old_state['category'] == 'payment' else 'human'

In [93]:
from langchain.agents import Tool

def get_order(order_id) :
    '''Use this to fetch order details from the mock DB'''
    print('Tool Called !')
    if order_id in orders:
        orderDetails = orders.get(order_id)
        print(f'found order details - {orderDetails}')
        return orderDetails
    else:
        return 'order not found'

orderTool = Tool(
    name='fetch_order_details',
    func=get_order,
    description='Use this to fetch order details from the mock DB'
)

tools = [orderTool]

worker_llm = ChatOpenAI(model='gpt-5-nano')
worker_llm_w_tools = worker_llm.bind_tools(tools)


In [94]:
orderTool

Tool(name='fetch_order_details', description='Use this to fetch order details from the mock DB', func=<function get_order at 0x0000026E05A776A0>)

In [ ]:
def orderAgent(old_state:State) :
    PROMPT = f''' 
    You are a order agent for an ecommerce platform. You are provided with a user query which is this \n
    {old_state['messages']}.
    You are also provided with a fetch order tool. use that tool to fetch the order data. If the order data is found,
    use that to answer the user's query. If the order is not found in db, YOU MUST REPLY rudely and angrily to the user.
    Answer ONLY on the basis the data returned by the tool.
    Do not invent actions, tracking information, or capabilities.
    '''
    orderResponse = worker_llm_w_tools.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    
    return {
        'messages' : [orderResponse]
    }

In [102]:
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder = StateGraph(State)
graph_builder.add_node('classifier', classifierAgent)
graph_builder.add_node('orderAgent', orderAgent)
graph_builder.add_node('tools', ToolNode(tools=tools))

graph_builder.add_edge(START, 'classifier')
graph_builder.add_conditional_edges('classifier', classify_route,{'order' : 'orderAgent'})
graph_builder.add_conditional_edges('orderAgent', tools_condition, {'tools':'tools','__end__' : END})
graph_builder.add_edge('tools', 'orderAgent')

graph = graph_builder.compile()


In [103]:
from IPython.display import display, Image
#display(Image(graph.get_graph().draw_mermaid_png()))

In [104]:
state = {
    'messages' : [HumanMessage(content='where is my order ORD1010 and when will it be delivered?')]
}
res = graph.invoke(state)
print(res['messages'][-1].content)

category : order
Tool Called !
found order details - {'customer_name': 'Meera Iyer', 'order_date': '2026-05-27', 'amount': 1199, 'payment': {'status': 'Success', 'method': 'Wallet', 'transaction_id': 'TXN1010', 'error_code': None}, 'shipping': {'order_status': 'Processing', 'tracking_number': None, 'estimated_delivery': '2026-06-04', 'delivery_date': None}, 'refund': {'status': 'Not Requested', 'reason': None, 'eligible': True}}
Hi Meera, here’s the latest on ORD1010:

- Status: Processing
- Order date: 2026-05-27
- Estimated delivery: 2026-06-04 (delivery date not yet available)
- Tracking number: not assigned yet
- Payment: Successful via Wallet (Transaction TXN1010)

Would you like me to notify you when it ships or if the delivery date changes?
